# Setup Jupyter Notebook

In [ ]:
import os
import sys

# Set jupyter to reload modules automatically so we can modify the code and see the changes without restarting the kernel
%load_ext autoreload
%autoreload 2

# Add the parent directory to the path so we can import the modules
parent_directory = os.path.abspath('..')
sys.path.append(parent_directory)
sys.path.append(parent_directory + '/src')

# Settings

In [ ]:

# General Dataset Settings
shape = "drill"

batch_size_inside = 16384
offset_shape_box = 0.1

fixed_n_outside_points_train = 32000000 
n_test_points = 500_000

maxvolume_tets_in  = 10000 # set to 10000 to disable
max_edge_length_tets_in  = 0.005
mesh_data_path = "../meshes/"

# Model Settings
layer_width_model=50
num_hidden_layer_model=5

# Inflation Settings
epsilon_inflation = 1e-6

# Training Settings
start_lr = 0.00125
lr_reduction_factor = 0.7
lambda_factor_increase = 1.3
lambda_factor_decrease = 0.96
epochs_per_lr = 300
lr_reduction_depth = 7

margin_in = 0.007     
gammas = [0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006] # extend as needed to match lr_reduction_depth

# Result Visualization Settings

# Zero Level Set
resolution_mesh = 400
render_offset = 0.2
batch_size_meshing = 50000

# Distance Field
axis_name = 'y'
axis_value = 0.0
n_points_vis_df = 1_000_000


# Create Datasets

In [ ]:
from data_loader import DataLoader
from src.tet_data_set import TetDataSetTetBased, TetDataSetRandom

# Loads Datasets from buffer if it exists, otherwise creates it. Therefore first run might take longer.
inside_data_set_train  = TetDataSetTetBased(mesh_data_path, inside_shape=True,  offset=None,   shape_name=shape, maxvolume=maxvolume_tets_in,  max_edge_length=max_edge_length_tets_in)
outside_data_set_train = TetDataSetRandom(mesh_data_path, are_points_inside_tet_mesh=False, n_points=fixed_n_outside_points_train, offset=offset_shape_box, shape_name=shape, train_test_val="train") 
outside_data_set_test = TetDataSetRandom(mesh_data_path, are_points_inside_tet_mesh=False, n_points=n_test_points, offset=offset_shape_box, shape_name=shape, train_test_val="test")

inside_data_loader_train  = DataLoader(inside_data_set_train,  shuffle=True, batch_size=batch_size_inside)
outside_data_loader_train = DataLoader(outside_data_set_train, shuffle=True, batch_size=0) # We set the batch size later

# Set batch size of outside data loader such that it has the same number of batches as the inside data loader while using all points
outside_data_loader_train.adapt_number_of_batches_to(inside_data_loader_train)

# Use inside batch size since it is a good number for decent speed for the test and validation datasets
batch_size_test = inside_data_loader_train.batch_size
outside_data_loader_test = DataLoader(outside_data_set_test, shuffle=False, batch_size=batch_size_test)

# Create Randomly Initialized Model

In [ ]:
from src.lipschitz_neural_net import LipschitzNeuralNet

model = LipschitzNeuralNet(layer_width_model, num_hidden_layer_model)

# Modifiy Model to Contain Shape

In [ ]:
model.inflate(inside_data_loader_train, epsilon_inflation)

# Calculate Margin for Outside

In [ ]:
from tetrahedralization_with_tetgen import create_simple_tet_mesh_from_off

num_outside_points = len(outside_data_set_train.points)
total_volume_definition_volume = outside_data_set_train.calculate_total_volume_definition_volume()

tet_mesh_simple = create_simple_tet_mesh_from_off(shape, mesh_data_path)
total_volume_shape = tet_mesh_simple.calculate_total_volume()

outside_volume = total_volume_definition_volume - total_volume_shape
outside_volume_per_point = outside_volume/num_outside_points

margin_out = (3/(4*3.141592653589793) * outside_volume_per_point)**(1/3)
    
print("Using margin_out: ", margin_out)

# Training

In [ ]:
from src.trainer import Trainer
from src.losses import HKR_Loss

loss = HKR_Loss(margin_in=margin_in, margin_out=margin_out)

trainer = Trainer(inside_data_loader=inside_data_loader_train, 
                  outside_data_loader_train=outside_data_loader_train, 
                  outside_data_loader_test=outside_data_loader_test,
                  loss=loss, 
                  lambda_factor_increase=lambda_factor_increase,
                  lambda_factor_decrease=lambda_factor_decrease,
                  outside_data_loader_val=None)

model = trainer.train(model, start_lr, epochs_per_lr, lr_reduction_factor, lr_reduction_depth, gammas=gammas)    

In [ ]:
# plot training statistics
trainer.plot_training(start_epoch=None, end_epoch=None)

# Visualize Resulting Model

## Visualize Zero Level Set

In [ ]:
mesh_output_path = "result.off"
model.reconstruct_surface_mesh(
    mesh_output_path,
    resolution=resolution_mesh,
    batch_size=batch_size_meshing,
    lower_point=inside_data_set_train.AABB_lower-render_offset,
    upper_point=inside_data_set_train.AABB_upper+render_offset,
)

In [ ]:
import pyvista as pv
import numpy as np

pv.set_jupyter_backend('trame')

def read_off(filepath):
    with open(filepath, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    i = 1 if lines[0] == 'OFF' else 0
    n_verts, n_faces, _ = map(int, lines[i].split())
    i += 1
    verts = np.array([list(map(float, lines[i + j].split())) for j in range(n_verts)])
    i += n_verts
    face_arr = []
    for j in range(n_faces):
        parts = list(map(int, lines[i + j].split()))
        face_arr.extend(parts)
    return pv.PolyData(verts, np.array(face_arr))

mesh = read_off(mesh_output_path)
plotter = pv.Plotter()
plotter.add_mesh(mesh, color="lightblue", show_edges=False)
plotter.show()


## Visualize Distance Field

In [ ]:
from src.visualization import visualize_distance_field_cross_section

visualize_distance_field_cross_section(
    model,
    lower_point=inside_data_set_train.AABB_lower,
    upper_point=inside_data_set_train.AABB_upper,
    axis=axis_name,
    axis_value=axis_value,
    n_points=n_points_vis_df,
)
